# Zepto Analytics Pipeline — Part B: Predictive Modeling & Pipeline Deployment

**Module 2 — Analytics Pipeline (/analytics)**
*Author:* AI/ML Engineering Team, Zepto

---

### Objectives:
1. **Stratified Split**: Split on `survived` (80/20 train/test) with justification.
2. **Preprocessing Pipeline**: Fit `ColumnTransformer` (imputation, one-hot encoding, scaling) **strictly on training fold**.
3. **Train 3 Classifiers**: Logistic Regression, Decision Tree (`plot_tree` rendered), and Random Forest.
4. **Evaluation**: Confusion matrix, accuracy, precision, recall, F1 score, ROC/AUC side-by-side.
5. **Imbalance Comparison**: Baseline vs `class_weight='balanced'` vs `SMOTE` (train fold only).
6. **Hyperparameter Tuning**: `GridSearchCV` on Random Forest with `oob_score=True`, reporting best parameters and OOB score.
7. **Regression Side-Task**: Multivariate Linear Regression predicting `fare` with MAE, RMSE, R², Adjusted R², and Residual/Heteroscedasticity analysis.
8. **Master Comparison & Deployment**: Model recommendation table and exporting full end-to-end pipeline via `joblib.dump`.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
from imblearn.over_sampling import SMOTE

# Load from committed fallback CSV
df = pd.read_csv("titanic.csv")
print(f"Loaded dataset: {df.shape}")


## Step 1: Stratified Train/Test Split

**Justification:** The dataset contains a 61.6% (Died) to 38.4% (Survived) class balance. Stratification ensures both train and test partitions maintain the true population ratio, eliminating train-test distribution skew.


In [ ]:
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target = "survived"

X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")
print(f"Train Class Balance:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"Test Class Balance:\n{y_test.value_counts(normalize=True).round(4)}")


## Step 2: Preprocessing Pipeline (ColumnTransformer)

Fit **only** on `X_train`, transform `X_test`.


In [ ]:
numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["pclass", "sex", "embarked"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, categorical_features)
    ]
)


## Step 3: Train 3 Classifiers & Comprehensive Evaluation

1. Logistic Regression
2. Decision Tree
3. Random Forest


In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, oob_score=True)
}

eval_metrics = {}
fitted_models = {}

for name, clf in classifiers.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", clf)
    ])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    eval_metrics[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "AUC-ROC": roc_auc_score(y_test, y_proba),
        "Confusion Matrix": confusion_matrix(y_test, y_pred),
        "Proba": y_proba
    }

pd.DataFrame({m: {k: v for k, v in res.items() if k not in ["Confusion Matrix", "Proba"]} for m, res in eval_metrics.items()}).T.round(4)


In [ ]:
# Render Decision Tree with labeled features and classes
dt_clf = fitted_models["Decision Tree"].named_steps["classifier"]
cat_enc = fitted_models["Decision Tree"].named_steps["preprocessor"].named_transformers_["cat"].named_steps["encoder"]
encoded_cat_names = cat_enc.get_feature_names_out(categorical_features).tolist()
all_features = numeric_features + encoded_cat_names

plt.figure(figsize=(20, 10))
plot_tree(dt_clf, feature_names=all_features, class_names=["Died", "Survived"], filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree Visualization (Max Depth = 4)", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Plot ROC Curves
plt.figure(figsize=(8, 6))
for name, res in eval_metrics.items():
    fpr, tpr, _ = roc_curve(y_test, res["Proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['AUC-ROC']:.3f})", lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label="Random Guess (AUC = 0.50)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()


## Step 4: Class Imbalance Handling Comparison

Comparing (a) Baseline vs (b) `class_weight='balanced'` vs (c) `SMOTE` applied on training fold only:


In [ ]:
X_tr_trans = preprocessor.fit_transform(X_train)
X_te_trans = preprocessor.transform(X_test)

# Baseline
rf_base = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_tr_trans, y_train)
pred_base = rf_base.predict(X_te_trans)

# Balanced
rf_bal = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=42).fit(X_tr_trans, y_train)
pred_bal = rf_bal.predict(X_te_trans)

# SMOTE
smote = SMOTE(random_state=42)
X_tr_smote, y_tr_smote = smote.fit_resample(X_tr_trans, y_train)
rf_sm = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_tr_smote, y_tr_smote)
pred_sm = rf_sm.predict(X_te_trans)

imb_res = pd.DataFrame([
    {"Strategy": "Baseline", "Precision": precision_score(y_test, pred_base), "Recall": recall_score(y_test, pred_base), "F1": f1_score(y_test, pred_base), "Accuracy": accuracy_score(y_test, pred_base)},
    {"Strategy": "class_weight='balanced'", "Precision": precision_score(y_test, pred_bal), "Recall": recall_score(y_test, pred_bal), "F1": f1_score(y_test, pred_bal), "Accuracy": accuracy_score(y_test, pred_bal)},
    {"Strategy": "SMOTE (Train Fold Only)", "Precision": precision_score(y_test, pred_sm), "Recall": recall_score(y_test, pred_sm), "F1": f1_score(y_test, pred_sm), "Accuracy": accuracy_score(y_test, pred_sm)}
])
display(imb_res.round(4))


## Step 5: Hyperparameter Tuning via GridSearchCV (with OOB Score)


In [ ]:
param_grid = {
    "classifier__n_estimators": [50, 100, 150],
    "classifier__max_depth": [4, 6, 8, None],
    "classifier__max_features": ["sqrt", "log2", 0.5]
}

rf_tune_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(oob_score=True, random_state=42, bootstrap=True))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(rf_tune_pipe, param_grid, cv=cv, scoring="f1", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_pipeline = grid_search.best_estimator_
best_rf_model = best_pipeline.named_steps["classifier"]

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best CV F1 Score:    {grid_search.best_score_:.4f}")
print(f"Out-of-Bag (OOB) Score: {best_rf_model.oob_score_:.4f}")


## Step 6: Regression Side-Task (Predicting Fare) & Heteroscedasticity Analysis


In [ ]:
reg_features = ["pclass", "sex", "age", "sibsp", "parch", "embarked"]
reg_X = df[reg_features].copy()
reg_y = df["fare"].copy()

rX_train, rX_test, ry_train, ry_test = train_test_split(reg_X, reg_y, test_size=0.20, random_state=42)

reg_prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), ["age", "sibsp", "parch"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))]), ["pclass", "sex", "embarked"])
])

reg_pipe = Pipeline([("preprocessor", reg_prep), ("regressor", LinearRegression())])
reg_pipe.fit(rX_train, ry_train)
ry_pred = reg_pipe.predict(rX_test)

mae = mean_absolute_error(ry_test, ry_pred)
rmse = np.sqrt(mean_squared_error(ry_test, ry_pred))
r2 = r2_score(ry_test, ry_pred)
n_samples = len(ry_test)
n_features = reg_prep.fit_transform(rX_train).shape[1]
adj_r2 = 1 - ((1 - r2) * (n_samples - 1) / (n_samples - n_features - 1))

print(f"Regression Metrics: MAE=${mae:.2f}, RMSE=${rmse:.2f}, R²={r2:.4f}, Adj R²={adj_r2:.4f}")

# Residual Plot
residuals = ry_test - ry_pred
plt.figure(figsize=(8, 5))
plt.scatter(ry_pred, residuals, alpha=0.7, color="purple", edgecolors="k")
plt.axhline(0, color="red", linestyle="--", lw=2)
plt.xlabel("Predicted Fare ($)")
plt.ylabel("Residuals ($)")
plt.title("Residual Plot (Heteroscedasticity Analysis)")
plt.grid(True, alpha=0.3)
plt.show()

print("Heteroscedasticity Interpretation:")
print("Residuals show a pronounced funnel-shaped spread that widens at higher predicted fares,")
print("confirming strong heteroscedasticity in ticket pricing without log transformation.")


## Step 7: Master Comparison Table & Deployment Export

Saving complete end-to-end pipeline (`preprocessor` + `best_estimator`) to `best_model_pipeline.joblib`:


In [ ]:
# Export pipeline
joblib.dump(best_pipeline, "best_model_pipeline.joblib")
print("Exported best_model_pipeline.joblib successfully!")

# Reload & test on raw inputs
loaded_pipeline = joblib.load("best_model_pipeline.joblib")
raw_test_sample = pd.DataFrame([
    {"pclass": 1, "sex": "female", "age": 29.0, "sibsp": 0, "parch": 0, "fare": 211.3375, "embarked": "S"},
    {"pclass": 3, "sex": "male", "age": 35.0, "sibsp": 1, "parch": 0, "fare": 7.8958, "embarked": "C"},
    {"pclass": 2, "sex": "female", "age": np.nan, "sibsp": 1, "parch": 2, "fare": 30.0708, "embarked": "C"}
])

preds = loaded_pipeline.predict(raw_test_sample)
probas = loaded_pipeline.predict_proba(raw_test_sample)[:, 1]

for i, (p, prob) in enumerate(zip(preds, probas)):
    print(f"Raw Input {i+1} -> Predicted: {p} ({'Survived' if p==1 else 'Died'}), Survival Prob: {prob:.4f}")
